Migrated from Databricks to Neuron (auto-port).

In [ ]:
import os
import sys
from pathlib import Path

for _candidate in (
    "/home/jovyan/work/default/rashmi-ranjan/lib",
    str(Path.cwd().parent / "lib"),
    str(Path.cwd() / "lib"),
):
    if _candidate not in sys.path and Path(_candidate).is_dir():
        sys.path.insert(0, _candidate)

from neuron_runtime import (
    GoogleSheetReader,
    SlackReportsTrigger,
    get_sql_df,
    load_service_account,
    load_slack_token,
    make_unique_columns,
)

service_account = load_service_account()
slack_token = load_slack_token()

In [ ]:
import trino
import pandas as pd


query = """

SELECT
    annotated_at_date AS "Date",
    SUM(gen_ai_canonical) AS "#Gen AI Canonical Tickets"
FROM (
    SELECT
        annotated_at_date,
        cluster_nfs_ntl,
        MAX(gen_ai_tickets) AS gen_ai_canonical
    FROM corp_analytics.process_excellence_qc_slack_card_tbl
    WHERE annotated_at_date >= CURRENT_DATE - INTERVAL '45' DAY
      AND annotated_at_date <= CURRENT_DATE - INTERVAL '1' DAY
      AND template_id = 177
      AND (ticket_group IS NULL OR ticket_group != 'PERSONAL_LOAN')
    GROUP BY annotated_at_date, cluster_nfs_ntl
)
GROUP BY annotated_at_date
ORDER BY annotated_at_date DESC
  
"""

df  = get_sql_df(query)
if not df.empty:
    display(df)

In [ ]:
# Python cell
import os
import shutil

if not os.path.exists("data"):
    os.makedirs("data")

gsheet_url = "https://docs.google.com/spreadsheets/d/1ZRuu6__cxniL_Uy_qzm7EekpcbhhuaxyowCT_z4vszY/edit?gid=1676905667#gid=1676905667"
worksheet_name = "ticket_count"
gs_reader = GoogleSheetReader(gsheet_url, service_account)

gs_reader.clear_sheets(
    worksheet_name,
    start_row=1,
    start_column=1
)

combined_df = pd.concat([df], ignore_index=True) 

gs_reader.set_df_in_sheets( 
worksheet_name, 
combined_df, 
include_headers=True, 
row=1, 
col=1
)